# Assignment 2

In this assigment, we will work with the *Adult* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/2/adult). Extract the data files into the subdirectory: `../05_src/data/adult/` (relative to `./05_src/`).

# Load the data

Assuming that the files `adult.data` and `adult.test` are in `../05_src/data/adult/`, then you can use the code below to load them.

In [4]:
import pandas as pd
columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
    'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week',
    'native-country', 'income'
]
adult_dt = (pd.read_csv('../../05_src/data/adult/adult.data', header = None, names = columns)
              .assign(income = lambda x: (x.income.str.strip() == '>50K')*1))


# Get X and Y

Create the features data frame and target data:

+ Create a dataframe `X` that holds the features (all columns that are not `income`).
+ Create a dataframe `Y` that holds the target data (`income`).
+ From `X` and `Y`, obtain the training and testing data sets:

    - Use a train-test split of 70-30%. 
    - Set the random state of the splitting function to 42.

In [ ]:
# STEP 1: Get X and Y

# Create the features DataFrame (X) by dropping the 'income' column
X = adult_dt.drop(columns=['income'])

# Create the target DataFrame (Y) by selecting the 'income' column
Y = adult_dt['income']

# Print the shapes of X and Y to verify
print("Shape of X:", X.shape)
print("Shape of Y:", Y.shape)

Shape of X: (32561, 14)
Shape of Y: (32561,)


In [6]:
# STEP 2: Split the Data

from sklearn.model_selection import train_test_split

# Split the data into training and testing sets with a 70-30 split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

# Print the shapes of the resulting datasets
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of Y_train:", Y_train.shape)
print("Shape of Y_test:", Y_test.shape)


Shape of X_train: (22792, 14)
Shape of X_test: (9769, 14)
Shape of Y_train: (22792,)
Shape of Y_test: (9769,)


## Random States

Please comment: 

+ What is the [random state](https://scikit-learn.org/stable/glossary.html#term-random_state) of the [splitting function](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)? 
+ Why is it [useful](https://en.wikipedia.org/wiki/Reproducibility)?

*(Comment here.)*

* What is the random state of the splitting function?
    * The random state of the splitting function is 42.

* Why is it useful?
    * The random state parameter is useful because it ensures reproducibility of the results. When you use a random state, you are setting a seed for the random number generator used in the train-test split. This means that every time you run the code with the same random state, you will get the same split of the data into training and testing sets. This is important for several reasons:

        * Consistency: Using a random state allows you to get consistent results across different runs of the code, which is crucial for debugging and comparing models.

        * Reproducibility: It ensures that others can replicate your results if they use the same data and random state. This is especially important in research and collaborative projects.
        
        * Fair Comparison: When comparing different models or approaches, having a fixed random state ensures that the comparison is fair because the data splits remain constant.

##### REFERENCES:

* Scikit-Learn Glossary: The glossary provides an explanation of the random_state parameter and its significance.
    * https://scikit-learn.org/stable/glossary.html#term-random-state

* Scikit-Learn User Guide on Controlling Randomness: This guide discusses how to control the randomness in various scikit-learn functions and why it is important.
    * https://scikit-learn.org/stable/common_pitfalls.html#controlling-randomness

# Preprocessing

Create a [Column Transformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) that treats the features as follows:

- Numerical variables

    * Apply [KNN-based imputation for completing missing values](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html):
        
        + Consider the 7 nearest neighbours.
        + Weight each neighbour by the inverse of its distance, causing closer neigbours to have more influence than more distant ones.
    * [Scale features using statistics that are robust to outliers](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html#sklearn.preprocessing.RobustScaler).

- Categorical variables: 
    
    * Apply a [simple imputation strategy](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html#sklearn.impute.SimpleImputer):

        + Use the most frequent value to complete missing values, also called the *mode*.

    * Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html):
        
        + Handle unknown labels if they exist.
        + Drop one column for binary variables.
    
    
The column transformer should look like this:

![](./images/assignment_2__column_transformer.png)

In [ ]:
# STEP 1: Import necessary libraries.
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [12]:
# STEP 2: Identify numerical and categorical features (columns).
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

In [13]:
# STEP 3: Create transformers for numerical and categorical preprocessing

numerical_transformer = Pipeline(steps=[
    ('imputer', KNNImputer(n_neighbors=7, weights='distance')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='if_binary'))
])

In [14]:
# STEP 4: Combine the transformers into a ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [15]:
# STEP 5: Fit the preprocessor on the training data
preprocessor.fit(X_train)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  KNNImputer(n_neighbors=7,
                                                             weights='distance')),
                                                 ('scaler', StandardScaler())]),
                                 Index(['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss',
       'hours-per-week'],
      dtype='object')),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(drop='if_binary',
                                                                handle_unknown='ignore'))]),
                                 Index(['workclass', 'education', 'marital-status', 'occupation',
       'relationship', 'race', 'sex', 'native-country'],
      dtype='object'))])

In [16]:
# STEP 6: Transform the training and testing data
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [17]:
# STEP 7: Print the shape of the transformed data
print("Shape of X_train_processed:", X_train_processed.shape)
print("Shape of X_test_processed:", X_test_processed.shape)

Shape of X_train_processed: (22792, 107)
Shape of X_test_processed: (9769, 107)


## Model Pipeline

Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `classifier` and assign a [`RandomForestClassifier()`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) to it.

The pipeline looks like this:

![](./images/assignment_2__pipeline.png)

In [18]:
# STEP 1: Import necessary libraries
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

In [19]:
# STEP 2: Create the model pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', RandomForestClassifier())
])

In [20]:
# STEP 3: Fit the pipeline on the training data
model_pipeline.fit(X_train, Y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   KNNImputer(n_neighbors=7,
                                                                              weights='distance')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss',
       'hours-per-week'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(drop='if_binary',
                                                                                 handle_unknown='ignore'))]),
                                                  Index(['workclass', 'education', 'marital-status', 'occupation',
       'relationship', 'race', 'sex', 'native-country'],
      dtype='object'))])),
                ('classifier', RandomForestClassifier())])

In [21]:
# STEP 4: Print the training score
print("Training score:", model_pipeline.score(X_train, Y_train))

Training score: 0.9998683748683749


# Cross-Validation

Evaluate the model pipeline using [`cross_validate()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html):

+ Measure the following [preformance metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#common-cases-predefined-values): negative log loss, ROC AUC, accuracy, and balanced accuracy.
+ Report the training and validation results. 
+ Use five folds.


In [ ]:
# import necessary libraries
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, log_loss, roc_auc_score, accuracy_score, balanced_accuracy_score
import numpy as np

In [ ]:
# STEP 1: Define the scoring metrics
scoring = {
    'neg_log_loss': make_scorer(log_loss, greater_is_better=False, needs_proba=True),
    'roc_auc': 'roc_auc',
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced_accuracy'
}

# STEP 2: Perform cross-validation
cv_results = cross_validate(
    model_pipeline, X_train, Y_train, cv=5, scoring=scoring,
    return_train_score=True, n_jobs=-1
)

# STEP 3: Convert results to a DataFrame
cv_results_df = pd.DataFrame(cv_results)

# STEP 4: Sort results by negative log loss of the test set
cv_results_df_sorted = cv_results_df.sort_values(by='test_neg_log_loss', ascending=False)

/opt/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/metrics/_scorer.py:610: FutureWarning: The `needs_threshold` and `needs_proba` parameter are deprecated in version 1.4 and will be removed in 1.6. You can either let `response_method` be `None` or set it to `predict` to preserve the same behaviour.
  warnings.warn(
/opt/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Display the fold-level results as a pandas data frame and sorted by negative log loss of the test (validation) set.

In [ ]:
# Display the sorted results
print("Cross-Validation Results (sorted by test_neg_log_loss):")
print(cv_results_df_sorted)

Cross-Validation Results (sorted by test_neg_log_loss):
    fit_time  score_time  test_neg_log_loss  train_neg_log_loss  test_roc_auc  \
0  32.150652    1.859438          -0.349118           -0.081855      0.904905   
2  32.106660    1.888509          -0.369228           -0.081716      0.901126   
3  33.395976    0.859227          -0.371040           -0.082090      0.906214   
4  18.687098    0.350439          -0.371046           -0.081428      0.903616   
1  31.990879    1.904992          -0.376037           -0.081595      0.901609   

   train_roc_auc  test_accuracy  train_accuracy  test_balanced_accuracy  \
0            1.0       0.849967        1.000000                0.773128   
2            1.0       0.849276        0.999945                0.768831   
3            1.0       0.861343        1.000000                0.785411   
4            1.0       0.856955        1.000000                0.778515   
1            1.0       0.842948        0.999945                0.760812   

   tra

Calculate the mean of each metric. 

In [25]:
# Calculate and display the mean of each metric
mean_results = cv_results_df.mean()
print("\nMean Cross-Validation Results:")
print(mean_results)


Mean Cross-Validation Results:
fit_time                   29.666253
score_time                  1.372521
test_neg_log_loss          -0.367294
train_neg_log_loss         -0.081737
test_roc_auc                0.903494
train_roc_auc               1.000000
test_accuracy               0.852098
train_accuracy              0.999978
test_balanced_accuracy      0.773339
train_balanced_accuracy     0.999970
dtype: float64


Calculate the same performance metrics (negative log loss, ROC AUC, accuracy, and balanced accuracy) using the testing data `X_test` and `Y_test`. Display results as a dictionary.

*Tip*: both, `roc_auc()` and `neg_log_loss()` will require prediction scores from `pipe.predict_proba()`. However, for `roc_auc()` you should only pass the last column `Y_pred_proba[:, 1]`. Use `Y_pred_proba` with `neg_log_loss()`.

In [26]:
# Evaluate the model pipeline on the test set
Y_pred_proba = model_pipeline.predict_proba(X_test)

test_metrics = {
    'neg_log_loss': log_loss(Y_test, Y_pred_proba),
    'roc_auc': roc_auc_score(Y_test, Y_pred_proba[:, 1]),
    'accuracy': accuracy_score(Y_test, model_pipeline.predict(X_test)),
    'balanced_accuracy': balanced_accuracy_score(Y_test, model_pipeline.predict(X_test))
}

print("\nTest Set Results:")
print(test_metrics)



Test Set Results:
{'neg_log_loss': 0.3962414746322275, 'roc_auc': 0.9002545668711202, 'accuracy': 0.8545398710205753, 'balanced_accuracy': 0.7749097581745152}


# Target Recoding

In the first code chunk of this document, we loaded the data and immediately recoded the target variable `income`. Why is this [convenient](https://scikit-learn.org/stable/modules/model_evaluation.html#binary-case)?

The specific line was:

```
adult_dt = (pd.read_csv('../05_src/data/adult/adult.data', header = None, names = columns)
              .assign(income = lambda x: (x.income.str.strip() == '>50K')*1))
```

(Answer here.)

This line of code reads the data from the specified CSV file and immediately recodes the target variable income. Specifically, it transforms the income column from categorical values ('>50K' and '<=50K') to binary numerical values (1 and 0, respectively). Here's why this step is convenient:

1. Simplifies Binary Classification: By converting the income column to binary numerical values (1 for '>50K' and 0 for '<=50K'), the problem is directly set up for binary classification. Many machine learning algorithms expect the target variable to be in a numerical format, especially for binary classification tasks.

2. Consistency in Data Representation: Recoding the target variable immediately ensures consistency in how the target variable is represented throughout the data processing and model training pipeline. This reduces the risk of errors or inconsistencies later in the workflow.

3. Streamlines Data Processing: Performing this transformation at the time of data loading consolidates the data preparation steps, making the code cleaner and easier to understand. It eliminates the need for a separate step to transform the target variable, thereby simplifying the overall data processing pipeline.

4. Efficiency: By incorporating the transformation into the data loading step, it makes the operation more efficient. The transformation is applied directly as the data is read into memory, saving the need for additional processing passes over the data.

5. Facilitates Integration with Scikit-Learn: Scikit-learn functions and estimators work seamlessly with numerical target variables. Transforming the target variable to a binary numerical format at the outset ensures compatibility with various Scikit-learn utilities and algorithms, such as cross-validation, performance metrics, and model evaluation.

Recoding the target variable income to binary numerical values immediately upon loading the data is a convenient step because it simplifies the binary classification task, ensures consistency in data representation, streamlines the data processing workflow, enhances efficiency, and facilitates seamless integration with Scikit-learn's machine learning tools.

#### REFERENCES:

Scikit-Learn Documentation:

* Target Variable in Classification: Provides details on how target variables are used in classification tasks.
    * https://scikit-learn.org/stable/glossary.html#term-target-variable

* ColumnTransformer: Discusses the use of ColumnTransformer for preprocessing pipelines.
    * https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html

Pandas Documentation:

* pandas.read_csv: Documentation on how to read CSV files with Pandas.
    * https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html

* pandas.DataFrame.assign: Details on the assign method, which is used to add new columns to DataFrames.
    * https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.assign.html

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Becker,Barry and Kohavi,Ronny. (1996). Adult. UCI Machine Learning Repository. https://doi.org/10.24432/C5XW20.